# CASCADE Training (Recreated)

This notebook recreates the Pyr CASCADE pipeline with:

1. Data prep from a CSV of cell links into CASCADE-style `*_mini.mat` files for multiple `f0` variants.
2. Leave-one-cell-out (LOCO) training and evaluation, grouping repeated recordings of the same cell across sessions/states.
3. Summary metrics and figures (`corr`, `relative_error`, `relative_bias`) per model variant.

It is designed to **resume** from existing artifacts under:

`Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\Pyr\cascade_training`


In [5]:
# Cell 1: Prepare CASCADE-style datasets + LOCO splits

from pathlib import Path
import os
import re
import json
import glob
import copy
import hashlib
import pickle
import numpy as np
import pandas as pd
import scipy.io as sio

# ---------- user inputs ----------
# Local defaults are kept for interactive Windows use. On SSH/cluster runs, override these with
# environment variables from run_cascade_training_cluster.sh.
CSV_LINKS_PATH = Path(os.environ.get(
    "CASCADE_TRAINING_CSV_LINKS_PATH",
    r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\Dendrites\PyrLowFR.csv",
))  # CSV containing cell links (Link/folder/cell_path/path)

BASE_OUT = Path(os.environ.get(
    "CASCADE_TRAINING_BASE_OUT",
    r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\Pyr\cascade_training",
))
CASCADE_REPO = Path(os.environ.get(
    "CASCADE_REPO",
    r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\code\cal_vol_soma\_cascade_ref_repo",
))

# DFF variants requested
VARIANT_SPECS = [
    ("f01_p8",  {"mode": "global_percentile",  "percentile": 8,  "rolling_sec": None}),
    ("f02_p10", {"mode": "global_percentile",  "percentile": 10, "rolling_sec": None}),
    ("f03_p15", {"mode": "global_percentile",  "percentile": 15, "rolling_sec": None}),
    ("f04_p20", {"mode": "global_percentile",  "percentile": 20, "rolling_sec": None}),
    ("f05_p8_rw6s",  {"mode": "rolling_percentile", "percentile": 8, "rolling_sec": 6.0}),
    ("f06_p8_rw15s", {"mode": "rolling_percentile", "percentile": 8, "rolling_sec": 15.0}),
]

DEFAULT_CAL_SR_HZ = 30.0
DEFAULT_VOL_SR_HZ = 500.0

# If False, existing variant mats are kept and missing files only are added.
FORCE_REWRITE_MATS = False
# If False, existing split folders are kept and only missing ones are added.
FORCE_REWRITE_SPLITS = False


# ---------- helpers ----------
def _norm_path(x):
    s = str(x).strip().replace('/', '\\')
    while '\\\\' in s:
        s = s.replace('\\\\', '\\')
    return s


def _norm_key(x):
    return _norm_path(x).lower()


def _first_present(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None


def _to_cell_dir(path_like):
    s = _norm_path(path_like)
    if s == '' or s.lower() == 'nan':
        return ''
    low = s.lower()
    suffixes = [
        'caltracenb.csv',
        'caltracenbdf.csv',
        'caltracenbdf..csv',
        'caltrace.csv',
        'voltrace.csv',
        'final_spikes.pkl',
    ]
    for suf in suffixes:
        if low.endswith('\\' + suf):
            return s[:-(len(suf) + 1)]
    if os.path.isfile(s):
        return os.path.dirname(s)
    return s


def _core_cell_id(cell_path):
    s = _norm_key(cell_path)
    toks = [t for t in s.split('\\') if t]
    if len(toks) < 2:
        return s

    fov_tok = None
    cell_tok = None
    for t in toks:
        if re.match(r'^fov\d+$', t):
            fov_tok = t
        if re.match(r'^cell\d+$', t):
            cell_tok = t

    date_idx = None
    for i, t in enumerate(toks):
        if re.match(r'^\d{2}-\d{2}-\d{4}($|-)', t):
            date_idx = i
            break

    if date_idx is not None and date_idx >= 2:
        animal_key = '\\'.join(toks[date_idx-2:date_idx])
    elif len(toks) >= 4:
        animal_key = '\\'.join(toks[-4:-2])
    else:
        animal_key = '\\'.join(toks[:-2])

    if fov_tok is None:
        fov_tok = toks[-2] if len(toks) >= 2 else 'fov?'
    if cell_tok is None:
        cell_tok = toks[-1] if len(toks) >= 1 else 'cell?'

    return f"{animal_key}|{fov_tok}|{cell_tok}"


def _safe_float(x, default):
    try:
        v = float(x)
        if np.isfinite(v):
            return v
    except Exception:
        pass
    return float(default)


def _read_csv_1d(path):
    arr = pd.read_csv(path).to_numpy(dtype=float).ravel()
    return arr


def _load_cal_trace_nb(cell_dir):
    p = os.path.join(cell_dir, 'calTraceNB.csv')
    if not os.path.isfile(p):
        return None
    try:
        return _read_csv_1d(p)
    except Exception:
        return None


def _dff_global_percentile(trace, percentile):
    x = np.asarray(trace, dtype=float).ravel()
    if x.size == 0:
        return x
    finite = np.isfinite(x)
    if not np.any(finite):
        return np.full_like(x, np.nan)
    f0 = np.nanpercentile(x[finite], float(percentile))
    if (not np.isfinite(f0)) or abs(f0) < 1e-12:
        f0 = 1e-12
    return (x - f0) / f0


def _dff_rolling_percentile(trace, percentile, cal_sr_hz, rolling_sec):
    x = np.asarray(trace, dtype=float).ravel()
    if x.size == 0:
        return x
    win = int(max(3, round(float(rolling_sec) * float(cal_sr_hz))))
    ser = pd.Series(x)
    f0 = ser.rolling(window=win, center=True, min_periods=max(3, win // 4)).quantile(float(percentile)/100.0)
    f0 = f0.to_numpy(dtype=float)

    # fill edges and non-finite entries
    if np.any(np.isfinite(f0)):
        idx = np.arange(f0.size)
        good = np.isfinite(f0)
        f0 = np.interp(idx, idx[good], f0[good])
    else:
        base = np.nanpercentile(x[np.isfinite(x)], float(percentile)) if np.any(np.isfinite(x)) else 1e-12
        f0 = np.full_like(x, base)

    f0[np.abs(f0) < 1e-12] = 1e-12
    return (x - f0) / f0


def _infer_vol_sr_hz(cell_dir, n_cal, cal_sr_hz):
    p = os.path.join(cell_dir, 'volTrace.csv')
    if os.path.isfile(p):
        try:
            v = _read_csv_1d(p)
            if v.size > 10 and n_cal > 10:
                return float(v.size) * float(cal_sr_hz) / float(n_cal)
        except Exception:
            pass
    return float(DEFAULT_VOL_SR_HZ)


def _state_order_token(tok):
    t = str(tok).lower()
    m = re.match(r'^[mr](\d+)$', t)
    if m:
        return (0 if t.startswith('m') else 1, int(m.group(1)))
    if t in ('', 'base', 'all'):
        return (2, 0)
    return (3, 0)


def _list_spike_pickles_by_priority(cell_dir):
    patterns = [
        (0, re.compile(r'^spike_detection_refined_new(?P<suf>[mr]\d+)?_rm_complex_highplateau\.pkl$', re.IGNORECASE)),
        (1, re.compile(r'^spike_detection_refined_new(?P<suf>[mr]\d+)?_rm_complex_after_peak\.pkl$', re.IGNORECASE)),
        (2, re.compile(r'^spike_detection_refined_new_plus_plateau(?P<suf>[mr]\d+)?\.pkl$', re.IGNORECASE)),
        (3, re.compile(r'^final_correct_spike_detection(?P<suf>[mr]\d+)?\.pkl$', re.IGNORECASE)),
        (4, re.compile(r'^spike_detection_refined_new(?P<suf>[mr]\d+)?\.pkl$', re.IGNORECASE)),
    ]

    cand = []
    for p in glob.glob(os.path.join(cell_dir, '*.pkl')):
        bn = os.path.basename(p)
        for rank, rgx in patterns:
            m = rgx.match(bn)
            if m:
                suf = m.group('suf') or ''
                cand.append((rank, suf.lower(), p))
                break

    # best per suffix
    best = {}
    for rank, suf, p in cand:
        prev = best.get(suf)
        if prev is None or rank < prev[0]:
            best[suf] = (rank, p)

    out = []
    for suf, (rank, p) in best.items():
        state_id = suf if suf else 'base'
        out.append((state_id, p))

    out.sort(key=lambda x: _state_order_token(x[0]))
    return out


def _load_spike_idx_from_pickle(pkl_path):
    try:
        with open(pkl_path, 'rb') as f:
            d = pickle.load(f)
    except ModuleNotFoundError as e:
        msg = str(e)
        if 'numpy._core' not in msg:
            return np.array([], dtype=int)
        try:
            import numpy.core as _np_core
            import sys as _sys
            _sys.modules.setdefault('numpy._core', _np_core)
            if hasattr(_np_core, 'multiarray'):
                _sys.modules.setdefault('numpy._core.multiarray', _np_core.multiarray)
            if hasattr(_np_core, 'numeric'):
                _sys.modules.setdefault('numpy._core.numeric', _np_core.numeric)
            with open(pkl_path, 'rb') as f:
                d = pickle.load(f)
        except Exception:
            return np.array([], dtype=int)
    except Exception:
        return np.array([], dtype=int)

    if not isinstance(d, dict):
        return np.array([], dtype=int)

    if 'vm_all_spikes' in d:
        sp = np.asarray(d.get('vm_all_spikes', []), dtype=float).ravel()
    else:
        sp = np.asarray(d.get('spike_indices', []), dtype=float).ravel()

    sp = sp[np.isfinite(sp)]
    if sp.size == 0:
        return np.array([], dtype=int)
    sp = np.unique(np.rint(sp).astype(int))
    return sp[sp >= 0]


def _make_cattached_mat(out_path, dff, cal_sr_hz, spike_idx, vol_sr_hz):
    dff = np.asarray(dff, dtype=float).ravel()
    n = int(dff.size)
    t = np.arange(n, dtype=float) / float(cal_sr_hz)
    events_ap = (np.asarray(spike_idx, dtype=float) / float(vol_sr_hz)) * 1e4

    # CASCADE-compatible trial layout: trial[0][0] must be a struct with fields.
    trial = np.zeros((1, 1), dtype=[('fluo_mean', 'O'), ('fluo_time', 'O'), ('events_AP', 'O')])
    trial[0, 0]['fluo_mean'] = dff
    trial[0, 0]['fluo_time'] = t
    trial[0, 0]['events_AP'] = events_ap

    attached = np.empty((1, 1), dtype=object)
    attached[0, 0] = trial
    sio.savemat(str(out_path), {'CAttached': attached}, do_compression=True)


def _link_or_copy(src, dst):
    src = Path(src)
    dst = Path(dst)
    if dst.exists():
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.link(str(src), str(dst))
    except Exception:
        import shutil
        shutil.copy2(str(src), str(dst))


# ---------- run ----------
if str(CSV_LINKS_PATH).strip() == '':
    raise RuntimeError('Set CSV_LINKS_PATH first (path to your link table CSV).')
if not CSV_LINKS_PATH.is_file():
    raise RuntimeError(f'CSV file not found: {CSV_LINKS_PATH}')

BASE_OUT.mkdir(parents=True, exist_ok=True)
(BASE_OUT / 'variants').mkdir(parents=True, exist_ok=True)
(BASE_OUT / 'splits_leave_one_cell').mkdir(parents=True, exist_ok=True)

print(f'Loading table: {CSV_LINKS_PATH}')
df = pd.read_csv(CSV_LINKS_PATH)

link_col = _first_present(df.columns, ['Link', 'folder', 'cell_path', 'path'])
if link_col is None:
    raise RuntimeError('CSV must include one of: Link, folder, cell_path, path')
cal_col = _first_present(df.columns, ['CALsr', 'CalSr', 'cal_sr', 'calsr'])

records_all = []
for ridx, row in df.iterrows():
    cell_dir = _to_cell_dir(row.get(link_col, ''))
    if cell_dir == '' or not os.path.isdir(cell_dir):
        continue

    cal_nb = _load_cal_trace_nb(cell_dir)
    if cal_nb is None or cal_nb.size < 10:
        continue

    cal_sr_hz = _safe_float(row.get(cal_col, DEFAULT_CAL_SR_HZ) if cal_col is not None else DEFAULT_CAL_SR_HZ, DEFAULT_CAL_SR_HZ)
    vol_sr_hz = _infer_vol_sr_hz(cell_dir, n_cal=int(cal_nb.size), cal_sr_hz=cal_sr_hz)

    spikes_by_state = _list_spike_pickles_by_priority(cell_dir)
    if len(spikes_by_state) == 0:
        continue

    core_id = _core_cell_id(cell_dir)
    cell_hash = hashlib.md5(_norm_key(cell_dir).encode('utf-8')).hexdigest()[:10]

    for state_id, pkl_path in spikes_by_state:
        spk_idx = _load_spike_idx_from_pickle(pkl_path)
        for variant_name, spec in VARIANT_SPECS:
            if spec['mode'] == 'global_percentile':
                dff = _dff_global_percentile(cal_nb, percentile=spec['percentile'])
            else:
                dff = _dff_rolling_percentile(
                    cal_nb,
                    percentile=spec['percentile'],
                    cal_sr_hz=cal_sr_hz,
                    rolling_sec=spec['rolling_sec'],
                )

            ds_name = f"DS00-CA1-{variant_name.split('_')[0]}"
            gt_dir = BASE_OUT / 'variants' / variant_name / 'Ground_truth' / ds_name
            gt_dir.mkdir(parents=True, exist_ok=True)

            mat_name = f"CAttached_{cell_hash}_{state_id}_mini.mat"
            mat_path = gt_dir / mat_name
            if FORCE_REWRITE_MATS or (not mat_path.is_file()):
                _make_cattached_mat(
                    out_path=mat_path,
                    dff=dff,
                    cal_sr_hz=cal_sr_hz,
                    spike_idx=spk_idx,
                    vol_sr_hz=vol_sr_hz,
                )

            records_all.append({
                'variant': variant_name,
                'dataset_name': ds_name,
                'cell_path': cell_dir,
                'core_cell_id': core_id,
                'state_id': state_id,
                'spike_pkl': pkl_path,
                'cal_sr_hz': cal_sr_hz,
                'vol_sr_hz': vol_sr_hz,
                'mat_path': str(mat_path),
                'mat_name': mat_name,
                'row_index': int(ridx),
            })

records_df = pd.DataFrame(records_all)
if len(records_df) == 0:
    raise RuntimeError('No valid records were prepared from CSV.')

records_csv = BASE_OUT / 'prepared_records_all.csv'
records_df.to_csv(records_csv, index=False)
print(f'Saved prepared record table: {records_csv}')

# Save per-variant manifest + create LOCO splits
for variant_name, _ in VARIANT_SPECS:
    vdf = records_df[records_df['variant'] == variant_name].copy()
    if len(vdf) == 0:
        print(f'[skip] no records for {variant_name}')
        continue

    var_dir = BASE_OUT / 'variants' / variant_name
    var_dir.mkdir(parents=True, exist_ok=True)
    var_manifest = var_dir / 'records_manifest.csv'
    vdf.to_csv(var_manifest, index=False)

    core_ids = sorted(vdf['core_cell_id'].dropna().astype(str).unique().tolist())
    split_root = BASE_OUT / 'splits_leave_one_cell' / variant_name
    split_root.mkdir(parents=True, exist_ok=True)

    for i, heldout in enumerate(core_ids, start=1):
        fold_id = f"fold_{i:03d}"
        fold_root = split_root / fold_id
        fold_root.mkdir(parents=True, exist_ok=True)

        train_df = vdf[vdf['core_cell_id'] != heldout].copy()
        test_df = vdf[vdf['core_cell_id'] == heldout].copy()

        if len(train_df) == 0 or len(test_df) == 0:
            continue

        train_csv = fold_root / 'train_records.csv'
        test_csv = fold_root / 'test_records.csv'
        train_df.to_csv(train_csv, index=False)
        test_df.to_csv(test_csv, index=False)

        ds_train_name = f"DS00_CA1_{variant_name}_train_fold{i:03d}"
        gt_fold_train = fold_root / 'Ground_truth' / ds_train_name

        if FORCE_REWRITE_SPLITS and gt_fold_train.exists():
            import shutil
            shutil.rmtree(gt_fold_train)

        gt_fold_train.mkdir(parents=True, exist_ok=True)

        # materialize train set as hardlinks/copies to save storage/time
        for mp in train_df['mat_path'].astype(str).tolist():
            src = Path(mp)
            if src.is_file():
                _link_or_copy(src, gt_fold_train / src.name)

        fold_meta = {
            'variant': variant_name,
            'fold_id': fold_id,
            'fold_index': i,
            'heldout_core_cell_id': heldout,
            'dataset_train_name': ds_train_name,
            'n_train_records': int(len(train_df)),
            'n_test_records': int(len(test_df)),
            'n_train_cells': int(train_df['core_cell_id'].nunique()),
            'n_test_cells': int(test_df['core_cell_id'].nunique()),
        }
        with open(fold_root / 'fold_meta.json', 'w', encoding='utf-8') as f:
            json.dump(fold_meta, f, indent=2)

    print(f'Prepared variant {variant_name}: {len(vdf)} recordings, {len(core_ids)} LOCO folds')

summary = {
    'csv_input': str(CSV_LINKS_PATH),
    'n_prepared_records': int(len(records_df)),
    'n_variants': int(records_df['variant'].nunique()),
    'variants': sorted(records_df['variant'].unique().tolist()),
}
with open(BASE_OUT / 'prep_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

print('Prep done.')
print(f'BASE_OUT: {BASE_OUT}')


Loading table: Z:\Adam-Lab-Shared\Data\Michal_Rubin\Dendrites\PyrLowFR.csv
Saved prepared record table: Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\Pyr\cascade_training\prepared_records_all.csv
Prepared variant f01_p8: 46 recordings, 29 LOCO folds
Prepared variant f02_p10: 46 recordings, 29 LOCO folds
Prepared variant f03_p15: 46 recordings, 29 LOCO folds
Prepared variant f04_p20: 46 recordings, 29 LOCO folds
Prepared variant f05_p8_rw6s: 46 recordings, 29 LOCO folds
Prepared variant f06_p8_rw15s: 46 recordings, 29 LOCO folds
Prep done.
BASE_OUT: Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\Pyr\cascade_training


In [ ]:
# Cell 2: LOCO training + evaluation (resume-aware)

from pathlib import Path
import os
import re
import sys
import json
import copy
import numpy as np
import pandas as pd
import scipy.io as sio
from scipy.ndimage import gaussian_filter1d

BASE_OUT = Path(os.environ.get(
    "CASCADE_TRAINING_BASE_OUT",
    r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\Pyr\cascade_training",
))
CASCADE_REPO = Path(os.environ.get(
    "CASCADE_REPO",
    r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\code\cal_vol_soma\_cascade_ref_repo",
))
MODEL_OUTPUT_ROOT = Path(os.environ.get("CASCADE_MODEL_OUTPUT_ROOT", str(BASE_OUT / "models_leave_one_cell")))
REPORT_OUTPUT_ROOT = Path(os.environ.get("CASCADE_REPORT_OUTPUT_ROOT", str(BASE_OUT / "reports_leave_one_cell")))

if str(CASCADE_REPO) not in sys.path:
    sys.path.insert(0, str(CASCADE_REPO))

from cascade2p import cascade, config

# training/eval config
VARIANTS_TO_RUN = [v.strip() for v in os.environ.get(
    "CASCADE_VARIANTS_TO_RUN",
    "f01_p8,f02_p10,f03_p15,f04_p20,f05_p8_rw6s,f06_p8_rw15s",
).split(",") if v.strip()]
FORCE_RETRAIN = os.environ.get("CASCADE_FORCE_RETRAIN", "0").strip().lower() in {"1", "true", "yes"}
FORCE_REEVAL = os.environ.get("CASCADE_FORCE_REEVAL", "0").strip().lower() in {"1", "true", "yes"}

MODEL_FAMILY = os.environ.get("CASCADE_MODEL_FAMILY", "GC8m_EX_30hz_smothing50ms_CA1")
SAMPLING_RATE_HZ = float(os.environ.get("CASCADE_SAMPLING_RATE_HZ", "30.0"))
SMOOTHING_S = float(os.environ.get("CASCADE_SMOOTHING_S", "0.05"))
NOISE_LEVELS = [int(x.strip()) for x in os.environ.get("CASCADE_NOISE_LEVELS", "2,4,6,8,10,14,19").split(",") if x.strip()]
ENSEMBLE_SIZE = int(os.environ.get("CASCADE_ENSEMBLE_SIZE", "5"))


def _variant_tag(variant_name):
    m = re.match(r'^(f\d+)', str(variant_name).lower())
    return m.group(1) if m else str(variant_name)


def _is_model_complete(model_dir):
    if not model_dir.is_dir():
        return False
    for nl in NOISE_LEVELS:
        for e in range(ENSEMBLE_SIZE):
            p = model_dir / f"Model_NoiseLevel_{nl}_Ensemble_{e}.h5"
            if not p.is_file():
                return False
    return True


def _expected_model_files(model_dir, noise_levels=None):
    if noise_levels is None:
        noise_levels = NOISE_LEVELS
    return [
        model_dir / f"Model_NoiseLevel_{nl}_Ensemble_{e}.h5"
        for nl in noise_levels
        for e in range(ENSEMBLE_SIZE)
    ]


def _missing_model_files(model_dir, noise_levels=None):
    return [p for p in _expected_model_files(model_dir, noise_levels=noise_levels) if not p.is_file()]


def _missing_noise_levels(model_dir):
    missing = []
    for nl in NOISE_LEVELS:
        if any(not (model_dir / f"Model_NoiseLevel_{nl}_Ensemble_{e}.h5").is_file() for e in range(ENSEMBLE_SIZE)):
            missing.append(nl)
    return missing


def _write_model_config(model_dir, cfg):
    model_dir.mkdir(parents=True, exist_ok=True)
    config.write_config(cfg, str(model_dir / 'config.yaml'))


def _load_cattached_recording(mat_path):
    data = sio.loadmat(str(mat_path))['CAttached'][0]
    trial = data[0]
    keys = trial[0][0].dtype.descr
    keys_unfolded = list(sum(keys, ()))

    traces_index = int(keys_unfolded.index('fluo_mean') / 2)
    fluo_time_index = int(keys_unfolded.index('fluo_time') / 2)
    events_index = int(keys_unfolded.index('events_AP') / 2)

    dff = np.squeeze(trial[0][0][traces_index]).astype(float)
    t = np.squeeze(trial[0][0][fluo_time_index]).astype(float)
    ev_ap = np.squeeze(trial[0][0][events_index]).astype(float)

    t = t[np.isfinite(t)]
    if len(t) < 2:
        raise RuntimeError(f'invalid fluo_time in {mat_path}')

    dff = dff[:len(t)]
    dff = dff[np.isfinite(t)]
    t = t[:len(dff)]

    fr = 1.0 / np.nanmean(np.diff(t))
    event_s = ev_ap[np.isfinite(ev_ap)] / 1e4
    return dff, t, event_s, float(fr)


def _events_to_fr(event_s, t):
    t = np.asarray(t, dtype=float).ravel()
    n = len(t)
    if n < 2:
        return np.array([], dtype=float)
    dt = np.nanmean(np.diff(t))
    fr = np.zeros(n, dtype=float)
    if event_s.size == 0:
        return fr

    edges = np.append(t, t[-1] + dt) - dt / 2.0
    counts, _ = np.histogram(event_s, bins=edges)
    fr = counts.astype(float) / dt
    return fr


def _pearson_corr_valid(a, b):
    a = np.asarray(a, dtype=float).ravel()
    b = np.asarray(b, dtype=float).ravel()
    n = min(a.size, b.size)
    if n <= 2:
        return np.nan
    a = a[:n]
    b = b[:n]
    ok = np.isfinite(a) & np.isfinite(b)
    if np.sum(ok) < 3:
        return np.nan
    aa = a[ok]
    bb = b[ok]
    if np.nanstd(aa) < 1e-12 or np.nanstd(bb) < 1e-12:
        return np.nan
    return float(np.corrcoef(aa, bb)[0, 1])


def _relative_error_bias_from_rates(true_fr_smooth, pred_fr_smooth, true_fr_raw, cal_sr):
    true = np.asarray(true_fr_smooth, dtype=float).ravel()
    pred = np.asarray(pred_fr_smooth, dtype=float).ravel()
    raw = np.asarray(true_fr_raw, dtype=float).ravel()
    n = min(true.size, pred.size, raw.size)
    if n <= 0:
        return np.nan, np.nan

    true = true[:n]
    pred = pred[:n]
    raw = raw[:n]

    ok = np.isfinite(true) & np.isfinite(pred) & np.isfinite(raw)
    if np.sum(ok) == 0:
        return np.nan, np.nan

    true_counts = np.clip(true[ok] / float(cal_sr), 0.0, None)
    pred_counts = np.clip(pred[ok] / float(cal_sr), 0.0, None)
    raw_true_counts = np.clip(raw[ok] / float(cal_sr), 0.0, None)

    denom = float(np.nansum(raw_true_counts))
    if (not np.isfinite(denom)) or denom <= 0:
        return np.nan, np.nan

    fp = float(np.nansum(np.clip(pred_counts - true_counts, 0.0, None)))
    fn = float(np.nansum(np.clip(true_counts - pred_counts, 0.0, None)))
    rel_error = (fp + fn) / denom
    rel_bias = (fp - fn) / denom
    return float(rel_error), float(rel_bias)


all_variant_rows = []

for variant in VARIANTS_TO_RUN:
    split_root = BASE_OUT / 'splits_leave_one_cell' / variant
    if not split_root.is_dir():
        print(f'[skip] missing split root: {split_root}')
        continue

    model_parent = MODEL_OUTPUT_ROOT / variant
    model_parent.mkdir(parents=True, exist_ok=True)

    report_dir = REPORT_OUTPUT_ROOT / variant
    report_dir.mkdir(parents=True, exist_ok=True)

    fold_dirs = sorted([p for p in split_root.glob('fold_*') if p.is_dir()])
    print(f'Variant {variant}: {len(fold_dirs)} folds')

    variant_rows = []

    for fold_dir in fold_dirs:
        fold_id = fold_dir.name
        fold_num = int(fold_id.split('_')[-1])

        meta_path = fold_dir / 'fold_meta.json'
        train_csv = fold_dir / 'train_records.csv'
        test_csv = fold_dir / 'test_records.csv'

        if not (meta_path.is_file() and train_csv.is_file() and test_csv.is_file()):
            print(f'  [skip {fold_id}] missing split files')
            continue

        with open(meta_path, 'r', encoding='utf-8') as f:
            meta = json.load(f)

        train_df = pd.read_csv(train_csv)
        test_df = pd.read_csv(test_csv)

        if len(train_df) == 0 or len(test_df) == 0:
            print(f'  [skip {fold_id}] empty train/test')
            continue

        var_tag = _variant_tag(variant)
        model_name = f"{MODEL_FAMILY}_{var_tag}_cellLOO_fold{fold_num:03d}"
        model_dir = model_parent / model_name

        # Build/update config
        cfg = dict(
            model_name=model_name,
            sampling_rate=float(SAMPLING_RATE_HZ),
            training_datasets=[meta['dataset_train_name']],
            noise_levels=copy.deepcopy(NOISE_LEVELS),
            smoothing=float(SMOOTHING_S),
            causal_kernel=0,
            windowsize=64,
            before_frac=0.5,
            filter_sizes=[31, 19, 5],
            filter_numbers=[30, 40, 50],
            dense_expansion=10,
            loss_function='mean_squared_error',
            optimizer='Adagrad',
            nr_of_epochs=20,
            ensemble_size=int(ENSEMBLE_SIZE),
            batch_size=1024,
            training_finished='No',
            verbose=1,
        )

        gt_root = fold_dir / 'Ground_truth'
        if not gt_root.is_dir():
            print(f'  [skip {fold_id}] missing Ground_truth folder')
            continue

        missing_before = _missing_model_files(model_dir)
        train_needed = FORCE_RETRAIN or (len(missing_before) > 0)
        if train_needed:
            if FORCE_RETRAIN:
                noise_to_train = copy.deepcopy(NOISE_LEVELS)
                print(f'  [train full] {model_name}: FORCE_RETRAIN=True')
            else:
                noise_to_train = _missing_noise_levels(model_dir)
                print(
                    f'  [resume train] {model_name}: '
                    f'{len(missing_before)}/{len(_expected_model_files(model_dir))} model files missing; '
                    f'training noise levels {noise_to_train}'
                )

            cfg_train = copy.deepcopy(cfg)
            cfg_train['noise_levels'] = copy.deepcopy(noise_to_train)
            cfg_train['training_finished'] = 'Running'
            _write_model_config(model_dir, cfg_train)

            cascade.train_model(model_name=model_name, model_folder=str(model_parent), ground_truth_folder=str(gt_root))

            missing_after = _missing_model_files(model_dir)
            cfg_done = copy.deepcopy(cfg)
            cfg_done['training_finished'] = 'Yes' if len(missing_after) == 0 else 'Running'
            _write_model_config(model_dir, cfg_done)

            if len(missing_after) > 0:
                print(f'  [incomplete after train] {model_name}: {len(missing_after)} files still missing')
                continue
            print(f'  [complete] {model_name}')
        else:
            cfg_done = copy.deepcopy(cfg)
            cfg_done['training_finished'] = 'Yes'
            _write_model_config(model_dir, cfg_done)
            print(f'  [reuse trained] {model_name}')

        fold_eval_csv = report_dir / f'{model_name}_test_metrics.csv'
        if fold_eval_csv.is_file() and (not FORCE_REEVAL):
            fold_rows = pd.read_csv(fold_eval_csv)
            variant_rows.extend(fold_rows.to_dict('records'))
            continue

        # Evaluate held-out recordings
        rec_rows = []
        true_all = []
        pred_all = []
        raw_all = []

        for _, r in test_df.iterrows():
            mat_path = Path(str(r['mat_path']))
            if not mat_path.is_file():
                continue

            try:
                dff, t, event_s, fr_hz = _load_cattached_recording(mat_path)
                pred = cascade.predict(
                    model_name=model_name,
                    traces=np.asarray(dff, dtype=float).reshape(1, -1),
                    model_folder=str(model_parent),
                    threshold=0,
                    verbosity=0,
                )
                pred = np.asarray(pred, dtype=float).reshape(-1)
                n = min(len(dff), len(pred), len(t))
                dff = dff[:n]
                t = t[:n]
                pred = pred[:n]

                pred_hz = pred * float(fr_hz)
                true_raw = _events_to_fr(event_s=event_s, t=t)

                sigma_frames = max(0.0, float(SMOOTHING_S) * float(fr_hz))
                if sigma_frames > 0:
                    true_s = gaussian_filter1d(true_raw.astype(float), sigma=sigma_frames, mode='nearest')
                    pred_s = gaussian_filter1d(pred_hz.astype(float), sigma=sigma_frames, mode='nearest')
                else:
                    true_s = true_raw.astype(float)
                    pred_s = pred_hz.astype(float)

                rec_corr = _pearson_corr_valid(true_s, pred_s)
                rec_re, rec_rb = _relative_error_bias_from_rates(true_s, pred_s, true_raw, fr_hz)

                rec_rows.append({
                    'variant': variant,
                    'fold_id': fold_id,
                    'model_name': model_name,
                    'heldout_core_cell_id': meta['heldout_core_cell_id'],
                    'cell_path': str(r.get('cell_path', '')),
                    'state_id': str(r.get('state_id', '')),
                    'mat_path': str(mat_path),
                    'corr_recording': rec_corr,
                    'relative_error_recording': rec_re,
                    'relative_bias_recording': rec_rb,
                })

                true_all.append(true_s)
                pred_all.append(pred_s)
                raw_all.append(true_raw)

            except Exception as e:
                rec_rows.append({
                    'variant': variant,
                    'fold_id': fold_id,
                    'model_name': model_name,
                    'heldout_core_cell_id': meta['heldout_core_cell_id'],
                    'cell_path': str(r.get('cell_path', '')),
                    'state_id': str(r.get('state_id', '')),
                    'mat_path': str(mat_path),
                    'corr_recording': np.nan,
                    'relative_error_recording': np.nan,
                    'relative_bias_recording': np.nan,
                    'error': str(e),
                })

        if len(true_all) > 0:
            true_cat = np.concatenate(true_all)
            pred_cat = np.concatenate(pred_all)
            raw_cat = np.concatenate(raw_all)
            fold_corr = _pearson_corr_valid(true_cat, pred_cat)
            fold_re, fold_rb = _relative_error_bias_from_rates(true_cat, pred_cat, raw_cat, SAMPLING_RATE_HZ)
        else:
            fold_corr, fold_re, fold_rb = np.nan, np.nan, np.nan

        fold_row = {
            'variant': variant,
            'fold_id': fold_id,
            'fold_index': fold_num,
            'model_name': model_name,
            'heldout_core_cell_id': meta['heldout_core_cell_id'],
            'n_test_records': int(len(test_df)),
            'corr': fold_corr,
            'relative_error': fold_re,
            'relative_bias': fold_rb,
        }
        variant_rows.append(fold_row)

        # Save per-recording details for this fold
        if len(rec_rows) > 0:
            pd.DataFrame(rec_rows).to_csv(report_dir / f'{model_name}_per_recording.csv', index=False)

        pd.DataFrame([fold_row]).to_csv(fold_eval_csv, index=False)

    if len(variant_rows) == 0:
        print(f'[warn] no fold rows for {variant}')
        continue

    vout = pd.DataFrame(variant_rows).sort_values('fold_index')
    vcsv = report_dir / f'{variant}_loco_metrics.csv'
    vout.to_csv(vcsv, index=False)
    print(f'Saved: {vcsv}')

    all_variant_rows.append(vout)

if len(all_variant_rows) == 0:
    raise RuntimeError('No variant metrics generated.')

all_df = pd.concat(all_variant_rows, ignore_index=True)
all_csv = REPORT_OUTPUT_ROOT / 'all_variants_loco_metrics.csv'
all_df.to_csv(all_csv, index=False)
print(f'All variants summary: {all_csv}')


Variant f01_p8: 29 folds
  [reuse trained] GC8m_EX_30hz_smothing50ms_CA1_f01_cellLOO_fold001
  [resume train] GC8m_EX_30hz_smothing50ms_CA1_f01_cellLOO_fold002: 2/35 model files missing; training noise levels [19]
Used configuration for model fitting (file Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\Pyr\cascade_training\models_leave_one_cell\f01_p8\GC8m_EX_30hz_smothing50ms_CA1_f01_cellLOO_fold002\config.yaml):

model_name:	GC8m_EX_30hz_smothing50ms_CA1_f01_cellLOO_fold002
sampling_rate:	30.0
training_datasets:	['DS00_CA1_f01_p8_train_fold002']
placeholder_1:	0
noise_levels:	[19]
placeholder_2:	0
smoothing:	0.05
causal_kernel:	0
windowsize:	64
before_frac:	0.5
filter_sizes:	[31, 19, 5]
filter_numbers:	[30, 40, 50]
dense_expansion:	10
loss_function:	mean_squared_error
optimizer:	Adagrad
nr_of_epochs:	20
ensemble_size:	5
batch_size:	1024
training_finished:	Running
verbose:	1


Models will be saved into this folder: Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\Pyr\cas

In [ ]:
# Cell 3: Summary plots (each held-out cell = one point)

from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

BASE_OUT = Path(os.environ.get(
    "CASCADE_TRAINING_BASE_OUT",
    r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\Pyr\cascade_training",
))
REPORT_OUTPUT_ROOT = Path(os.environ.get("CASCADE_REPORT_OUTPUT_ROOT", str(BASE_OUT / "reports_leave_one_cell")))
all_csv = REPORT_OUTPUT_ROOT / 'all_variants_loco_metrics.csv'
if not all_csv.is_file():
    raise RuntimeError(f'Missing file: {all_csv}. Run Cell 2 first.')

df = pd.read_csv(all_csv)
if len(df) == 0:
    raise RuntimeError('No rows in all_variants_loco_metrics.csv')

metrics = ['corr', 'relative_error', 'relative_bias']
metric_titles = {
    'corr': 'Correlation (real FR vs predicted FR)',
    'relative_error': 'Relative Error',
    'relative_bias': 'Relative Bias',
}

variants = sorted(df['variant'].astype(str).unique().tolist())

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=[metric_titles[m] for m in metrics],
    horizontal_spacing=0.08,
)

palette = ['#4C78A8', '#F58518', '#54A24B', '#E45756', '#72B7B2', '#B279A2', '#FF9DA6', '#9D755D']
color_map = {v: palette[i % len(palette)] for i, v in enumerate(variants)}

for j, m in enumerate(metrics, start=1):
    for v in variants:
        yy = pd.to_numeric(df.loc[df['variant'] == v, m], errors='coerce').to_numpy(dtype=float)
        yy = yy[np.isfinite(yy)]
        xx = np.array([v] * len(yy), dtype=object)

        fig.add_trace(
            go.Box(
                x=xx,
                y=yy,
                name=v,
                marker_color=color_map[v],
                boxpoints='all',
                jitter=0.25,
                pointpos=0.0,
                boxmean=True,
                width=0.6,
                opacity=0.85,
                showlegend=(j == 1),
            ),
            row=1,
            col=j,
        )

    fig.update_xaxes(categoryorder='array', categoryarray=variants, tickangle=45, row=1, col=j)

fig.update_layout(
    template='simple_white',
    width=2200,
    height=760,
    boxmode='group',
    title='CASCADE LOCO Performance Across f0 Variants',
)

out_dir = REPORT_OUTPUT_ROOT
out_dir.mkdir(parents=True, exist_ok=True)
html_path = out_dir / 'all_variants_loco_summary.html'
svg_path = out_dir / 'all_variants_loco_summary.svg'
png_path = out_dir / 'all_variants_loco_summary.png'
fig.write_html(html_path)
try:
    fig.write_image(svg_path)
except Exception:
    pass
try:
    fig.write_image(png_path)
except Exception:
    pass

print(f'Saved summary figure: {html_path}')
print(f'SVG: {svg_path}')
print(f'PNG: {png_path}')
